# 🏗️ Production-Grade Data Engineering with Python
## From Raw Ingestion to Polished Excel — A FAANG-Style Masterclass

**Audience:** Intermediate-to-advanced analysts / data engineers who need to go
beyond "read → clean → save" and handle the edge cases that blow up real pipelines.

### How every section works

| Step | What you get |
|---|---|
| **Concept & FAANG context** | *Why* this matters at scale — not just *what* the function does. |
| **Production-ready code** | Fully runnable, vectorized, with heavy inline comments on edge cases. |
| **Practice challenge** | A `# TODO` scenario with hints. Solutions are in the appendix at the end. |

### Libraries used
`pandas · numpy · openpyxl · xlsxwriter · rapidfuzz · duckdb`

> The notebook generates its own realistic, messy synthetic data on first run so it works
> out-of-the-box. Every `DATA_PATH` variable at the top of a cell is a placeholder you can
> swap to your own files later.

---
## ⚙️ Setup: Generate realistic messy datasets

We build several files that simulate the kind of exports you'd get from real business systems:

- **`raw_transactions.xlsx`** — 2 000 rows of messy transactional data across two sheets
  (`Jan_Mar` and `Apr_Jun`), complete with banner rows, mixed date formats, hidden nulls
  (`"N/A"`, `"-"`, empty strings), inconsistent casing, duplicates, and outlier amounts.
- **`customer_master.csv`** — a reference table with fuzzy/misspelled company names.
- **`product_catalog.json`** — a product lookup with pricing data.
- **`big_events.parquet`** — a 500k-row Parquet file for DuckDB speed demos.

Run this cell once. Everything downstream reads from disk.

In [1]:
import numpy as np
import pandas as pd
import json, os, random, string
from datetime import datetime, timedelta

np.random.seed(2026)
random.seed(2026)

def rand_date(start, end):
    delta = (end - start).days
    return start + timedelta(days=random.randint(0, delta))

def messy_date(d):
    roll = random.random()
    if roll < 0.03: return random.choice(["N/A", "-", "", "TBD", "null"])
    fmt = random.choice(["%m/%d/%Y", "%d-%b-%Y", "%Y-%m-%d", "%d/%m/%Y"])
    return d.strftime(fmt)

def messy_amount(base):
    if random.random() < 0.03: return random.choice(["N/A", "-", ""])
    if random.random() < 0.02: return str(round(base * random.uniform(50, 200), 2))
    return str(round(base * random.uniform(0.5, 2.0), 2))

regions = ["East", "West", "North", "South", "Central"]
categories = ["Electronics", "Furniture", "Office Supplies", "Software", "Services"]
payment_methods = ["Credit Card", "Wire Transfer", "ACH", "Check", "PayPal"]
statuses = ["Completed", "Pending", "Refunded", "Cancelled"]

clean_companies = [
    "Acme Corp", "Globex International", "Initech Solutions", "Umbrella Holdings",
    "Stark Industries", "Wayne Enterprises", "Soylent Corp", "Cyberdyne Systems",
    "Oscorp Technologies", "Aperture Science", "Hooli Inc", "Pied Piper",
    "Massive Dynamic", "Tyrell Corporation", "Weyland-Yutani", "Abstergo Industries",
    "Vought International", "LexCorp", "Dunder Mifflin", "Prestige Worldwide",
]
fuzzy_companies = {c: [c, c.lower(), c.replace(" ", "  "), c[:len(c)//2+3] + c[len(c)//2+3:].upper(),
                       c + " " ] for c in clean_companies}

cust_ids = [f"CUST-{str(i).zfill(4)}" for i in range(1, 201)]
product_codes = [f"SKU-{random.choice(string.ascii_uppercase)}{random.randint(100,999)}" for _ in range(40)]
product_codes = list(set(product_codes))[:30]

def make_txn_rows(n, date_start, date_end):
    rows = []
    for i in range(n):
        d = rand_date(date_start, date_end)
        company = random.choice(clean_companies)
        co_name = random.choice(fuzzy_companies[company]) if random.random() < 0.3 else company
        if random.random() < 0.15: co_name = co_name.upper()
        if random.random() < 0.10: co_name = co_name.lower()
        region = random.choice(regions)
        if random.random() < 0.08: region = random.choice([region.upper(), f" {region}", f"{region} ", region.lower()])
        rep = random.choice(["Alice", "Bob", "Carol", "Dan", "Eve", "Frank", np.nan])
        if isinstance(rep, str) and random.random() < 0.05: rep = f"  {rep} "
        rows.append({
            " TXN_ID": f"TXN-{100000+len(rows)+i}",
            "Order Date ": messy_date(d),
            "Customer_ID ": random.choice(cust_ids),
            " Company Name": co_name,
            "Product_Code": random.choice(product_codes),
            "Qty": random.randint(1, 50) if random.random() > 0.04 else random.choice(["", "N/A", np.nan]),
            " Amount (USD)": messy_amount(random.uniform(10, 5000)),
            "Region ": region,
            " Payment Method": random.choice(payment_methods),
            "Status": random.choice(statuses),
            "  Sales Rep  ": rep,
            "Notes\t": random.choice(["", "Rush", "Bulk discount", np.nan, "Follow up", "\t", "null"]),
        })
    return pd.DataFrame(rows)

df_q1 = make_txn_rows(1000, datetime(2026, 1, 1), datetime(2026, 3, 31))
df_q2 = make_txn_rows(1000, datetime(2026, 4, 1), datetime(2026, 6, 30))

with pd.ExcelWriter("raw_transactions.xlsx", engine="openpyxl") as w:
    df_q1.to_excel(w, sheet_name="Jan_Mar", index=False, startrow=2)
    df_q2.to_excel(w, sheet_name="Apr_Jun", index=False, startrow=0)

from openpyxl import load_workbook
wb = load_workbook("raw_transactions.xlsx")
ws = wb["Jan_Mar"]
ws["A1"] = "FINANCE SYSTEM EXPORT -- Q1 2026 -- CONFIDENTIAL -- DO NOT EDIT"
ws["A2"] = ""
wb.save("raw_transactions.xlsx")

cust_master_rows = []
for i, cid in enumerate(cust_ids):
    co = clean_companies[i % len(clean_companies)]
    cust_master_rows.append({
        "CustID": cid,
        "CompanyName": co,
        "Segment": random.choice(["Enterprise", "SMB", "Startup", "Government"]),
        "Country": random.choice(["US", "UK", "DE", "FR", "KE", "NG", "RW", "IN", "JP", "BR"]),
        "CreditLimit": round(random.uniform(5000, 500000), 2),
    })
pd.DataFrame(cust_master_rows).to_csv("customer_master.csv", index=False, encoding="utf-8")

prod_catalog = []
for pc in product_codes:
    prod_catalog.append({
        "ProductCode": pc,
        "ProductName": f"Widget {pc[-3:]}",
        "Category": random.choice(categories),
        "UnitPrice": round(random.uniform(5, 2000), 2),
        "Weight_kg": round(random.uniform(0.1, 50), 1),
    })
with open("product_catalog.json", "w") as f:
    json.dump(prod_catalog, f, indent=2)

big_n = 500_000
big_df = pd.DataFrame({
    "event_id": np.arange(big_n),
    "user_id": np.random.randint(1, 50001, big_n),
    "event_type": np.random.choice(["click", "view", "purchase", "signup", "logout"], big_n),
    "amount": np.round(np.random.exponential(50, big_n), 2),
    "ts": pd.date_range("2026-01-01", periods=big_n, freq="min"),
})
big_df.to_parquet("big_events.parquet", index=False)

print("Files created:")
for fn in ["raw_transactions.xlsx", "customer_master.csv", "product_catalog.json", "big_events.parquet"]:
    print(f"   {fn}  ({os.path.getsize(fn)/1024:.0f} KB)")

Files created:
   raw_transactions.xlsx  (134 KB)
   customer_master.csv  (9 KB)
   product_catalog.json  (4 KB)
   big_events.parquet  (7827 KB)


---
# Module 0: Diagnostic Data Profiling & Quality Assessment

> *"The first thing any new data source gets is a profiling pass -- before a single
> transformation is written. If you don't know what's broken, you can't write code that won't
> break when the next file arrives."*

Before touching a row, profile it. This module builds a reusable profiling toolkit that works
on *any* DataFrame, regardless of schema.

## 0.1 Automated profiling routine

**Concept:** A good profiling pass answers five questions in under a second: how big is this,
what types did pandas infer (and are they correct), how much is missing and where, how unique
each column is (cardinality), and what the distribution looks like.

**FAANG context:** At scale, `.info(memory_usage='deep')` reveals whether a column is burning 8x
the RAM it needs. Cardinality analysis catches columns that *look* categorical but are actually
near-unique IDs, which would explode a pivot.

In [2]:
import pandas as pd
import numpy as np

DATA_PATH = "raw_transactions.xlsx"
df_raw = pd.read_excel(DATA_PATH, sheet_name=0, skiprows=2)

def profile_dataframe(df, name="DataFrame"):
    # Production profiling routine: returns a summary DF and prints a human-readable report.
    print(f"{'='*60}")
    print(f"  PROFILE: {name}")
    print(f"{'='*60}")
    print(f"  Rows: {len(df):,}  |  Columns: {len(df.columns)}")
    mem = df.memory_usage(deep=True)
    print(f"  Total memory: {mem.sum() / 1024**2:.2f} MB")
    print()

    records = []
    for col in df.columns:
        s = df[col]
        records.append({
            "column": col,
            "dtype": str(s.dtype),
            "non_null": s.notna().sum(),
            "null_count": s.isna().sum(),
            "null_pct": round(s.isna().mean() * 100, 2),
            "n_unique": s.nunique(),
            "cardinality_pct": round(s.nunique() / max(len(s), 1) * 100, 2),
            "top_value": s.mode().iloc[0] if len(s.mode()) > 0 else None,
            "sample_values": str(s.dropna().unique()[:3].tolist()),
            "mem_MB": round(mem[col] / 1024**2, 4),
        })
    summary = pd.DataFrame(records)
    print(summary.to_string(index=False))
    return summary

profile_summary = profile_dataframe(df_raw, "raw_transactions / Jan_Mar")

  PROFILE: raw_transactions / Jan_Mar
  Rows: 1,000  |  Columns: 12
  Total memory: 0.18 MB

         column   dtype  non_null  null_count  null_pct  n_unique  cardinality_pct           top_value                                                    sample_values  mem_MB
         TXN_ID     str      1000           0       0.0      1000            100.0          TXN-100000                       ['TXN-100000', 'TXN-100002', 'TXN-100004']  0.0172
    Order Date      str       982          18       1.8       332             33.2          02/01/2026                      ['18-Mar-2026', '16/01/2026', '2026-01-26']  0.0173
   Customer_ID      str      1000           0       0.0       198             19.8           CUST-0082                          ['CUST-0197', 'CUST-0103', 'CUST-0082']  0.0162
   Company Name     str      1000           0       0.0       135             13.5 Oscorp Technologies ['Abstergo Industries', 'Weyland-Yutani', 'Oscorp Technologies']  0.0222
   Product_Code     str    

### 🧩 Practice 0.1
Load `customer_master.csv` and run `profile_dataframe()` on it. Before looking at the output,
predict: which column will have the **highest cardinality** and which will have the **lowest**?

**Hints:** `CustID` should be 100% unique (primary key). `Country` has only ~10 values.

In [3]:
# TODO: load customer_master.csv and profile it
# df_cust = pd.read_csv("customer_master.csv")
# profile_dataframe(df_cust, "Customer Master")

## 0.2 Missingness & hidden-null detection

**Concept:** `df.isna()` only catches `None` and `np.nan`. Real exports are full of **hidden nulls**:
strings like `"N/A"`, `"-"`, `"null"`, `""`, `"\t"`, and the invisible non-breaking space `\xa0`.
These pass through `.isna()` silently and break type casts, sums, and joins downstream.

**FAANG context:** Data quality checks run a "hidden null scan" on every ingested DataFrame
*before* any transform. The pattern: define a set of sentinel values, replace them with `np.nan`
in one vectorized pass, then re-run null counts.

In [4]:
HIDDEN_NULL_SENTINELS = {"", "N/A", "n/a", "NA", "na", "-", "--", "null", "NULL",
                          "none", "None", "NONE", ".", "..", "?", "TBD", "tbd",
                          "\t", "\xa0", " "}

def scan_hidden_nulls(df):
    # Scan every object column for sentinel values that pandas did NOT flag as NaN.
    results = []
    for col in df.columns:
        s = df[col]
        if s.dtype == object:
            stripped = s.astype(str).str.strip()
            mask = stripped.isin(HIDDEN_NULL_SENTINELS)
            hidden_count = mask.sum()
            if hidden_count > 0:
                found_vals = stripped[mask].value_counts().to_dict()
                results.append({"column": col, "hidden_null_count": hidden_count,
                                "sentinel_values_found": found_vals})
    report = pd.DataFrame(results)
    if report.empty:
        print("No hidden nulls detected.")
    else:
        print(f"Found hidden nulls in {len(report)} column(s):")
        print(report.to_string(index=False))
    return report

hidden_report = scan_hidden_nulls(df_raw)

def replace_hidden_nulls(df):
    # Replace all hidden-null sentinels with np.nan across every object column.
    df = df.copy()
    for col in df.select_dtypes(include="object").columns:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(HIDDEN_NULL_SENTINELS, np.nan)
        df[col] = df[col].replace("nan", np.nan)
    return df

df_clean_nulls = replace_hidden_nulls(df_raw)
print("\nNull counts AFTER hidden-null replacement:")
print(df_clean_nulls.isna().sum())

No hidden nulls detected.



Null counts AFTER hidden-null replacement:
 TXN_ID              0
Order Date          30
Customer_ID          0
 Company Name        0
Product_Code         0
Qty                 52
 Amount (USD)       27
Region               0
 Payment Method      0
Status               0
  Sales Rep        144
Notes\t            575
dtype: int64


/tmp/ipykernel_590/1342759540.py:31: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include="object").columns:


### 🧩 Practice 0.2
After running `replace_hidden_nulls`, confirm that the null count for the `" Amount (USD)"`
column increased compared to the raw data. Store the *difference* in `amount_null_increase`.

**Hints:**
- `df_raw[" Amount (USD)"].isna().sum()` vs `df_clean_nulls[" Amount (USD)"].isna().sum()`

In [5]:
# TODO: compute amount_null_increase
# amount_null_increase = ...

## 0.3 Memory usage profiling

**Concept:** `df.memory_usage(deep=True)` counts the actual bytes held by object columns, not
just the 8-byte pointer pandas stores by default. Deep mode can reveal 10x differences for
string-heavy DataFrames.

**FAANG context:** Memory profiling is how you decide whether a dataset should be processed
in-memory (pandas), out-of-core (DuckDB/Dask), or pushed to Spark.

In [6]:
print("Shallow memory estimate (misleading for string columns):")
print(f"  {df_raw.memory_usage(deep=False).sum() / 1024**2:.2f} MB")

print("\nDeep memory estimate (accurate):")
print(f"  {df_raw.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

mem_breakdown = (
    df_raw.memory_usage(deep=True)
    .drop("Index")
    .sort_values(ascending=False)
    .to_frame("bytes")
    .assign(MB=lambda x: (x["bytes"] / 1024**2).round(4))
)
print("\nPer-column memory (deep):")
print(mem_breakdown)

Shallow memory estimate (misleading for string columns):
  0.18 MB

Deep memory estimate (accurate):
  0.18 MB

Per-column memory (deep):
                 bytes      MB
 Company Name    23258  0.0222
Order Date       18116  0.0173
 TXN_ID          18000  0.0172
Customer_ID      17000  0.0162
Status           16220  0.0155
Product_Code     16000  0.0153
 Payment Method  15617  0.0149
 Amount (USD)    14714  0.0140
Region           13013  0.0124
Notes\t          12065  0.0115
  Sales Rep      11653  0.0111
Qty               8000  0.0076


### 🧩 Practice 0.3
Load `big_events.parquet` and compare its deep memory usage to the file size on disk. Store the
ratio (in-memory / file-size) in `memory_to_disk_ratio`.

**Hints:**
- `os.path.getsize(path)` gives file size in bytes.
- `df.memory_usage(deep=True).sum()` gives in-memory size in bytes.

In [7]:
import os
# TODO: load big_events.parquet, compute memory_to_disk_ratio
# memory_to_disk_ratio = ...

---
# Module 1: Production Ingestion & Boundary Handling

Getting data *into* pandas correctly is harder than it looks. This module covers the edge cases
that cause silent data loss or pipeline crashes in real workbooks: unknown sheet layouts, banner
rows, encoding landmines, and memory-safe reading of large files.

## 1.1 Multi-sheet & complex workbooks

**Concept:** Real workbooks have sheets you didn't expect, sheets with different header offsets,
and even sheets with no data at all. A production reader iterates over `wb.sheetnames`
dynamically rather than hard-coding `sheet_name="Sheet1"`. For sheets with banner rows, a
header-detection heuristic (find the first row where most cells are non-empty strings) is more
robust than guessing `skiprows=2`.

**FAANG context:** At Apple, finance exports rotate their sheet structure quarterly. Pipelines
that hard-code sheet names break every quarter; pipelines that discover and iterate don't.

In [8]:
from openpyxl import load_workbook
import pandas as pd

DATA_PATH = "raw_transactions.xlsx"

# Step 1: discover sheets without loading data (read_only=True is fast and light)
wb_meta = load_workbook(DATA_PATH, read_only=True)
print("Sheets found:", wb_meta.sheetnames)
wb_meta.close()

def detect_header_row(path, sheet_name, max_scan=10):
    # Heuristic: the header row is the first row where >= 50% of cells are non-empty strings.
    # This beats hard-coding skiprows, which breaks when the vendor changes the banner layout.
    raw = pd.read_excel(path, sheet_name=sheet_name, header=None, nrows=max_scan)
    for i, row in raw.iterrows():
        non_empty = row.dropna().astype(str).str.strip().replace("", pd.NA).dropna()
        if len(non_empty) >= len(row) * 0.5:
            return i
    return 0  # fallback: assume row 0 is the header

# Step 2: load every sheet dynamically, auto-detecting headers
all_dfs = {}
for sheet in wb_meta.sheetnames:
    hdr = detect_header_row(DATA_PATH, sheet)
    df = pd.read_excel(DATA_PATH, sheet_name=sheet, header=hdr)
    all_dfs[sheet] = df
    print(f"  {sheet}: header at row {hdr}, shape {df.shape}")

# Step 3: stack sheets with identical schemas into one DataFrame
df_stacked = pd.concat(all_dfs.values(), ignore_index=True)
print(f"\nStacked shape: {df_stacked.shape}")
df_stacked.head(3)

Sheets found: ['Jan_Mar', 'Apr_Jun']


  Jan_Mar: header at row 2, shape (1000, 12)


  Apr_Jun: header at row 0, shape (1000, 12)

Stacked shape: (2000, 12)


,TXN_ID,Order Date,Customer_ID,Company Name,Product_Code,Qty,Amount (USD),Region,Payment Method,Status,Sales Rep,Notes\t
0,TXN-100000,18-Mar-2026,CUST-0197,Abstergo Industries,SKU-N192,46.0,2998.55,Central,Wire Transfer,Cancelled,Dan,Rush
1,TXN-100002,16/01/2026,CUST-0103,Weyland-Yutani,SKU-U820,18.0,109.26,Central,PayPal,Cancelled,Dan,Bulk discount
2,TXN-100004,2026-01-26,CUST-0082,Oscorp Technologies,SKU-D560,7.0,1952.25,South,Wire Transfer,Cancelled,Eve,NaN


### 🧩 Practice 1.1
Write a function `load_all_sheets(path)` that: (a) discovers all sheet names, (b) auto-detects
the header row for each, (c) loads and stacks them, and (d) adds a `_source_sheet` column so
you know which sheet each row came from. Return the stacked DataFrame. Test it on
`raw_transactions.xlsx`.

**Hints:**
- In the `pd.concat` loop, do `df["_source_sheet"] = sheet_name` before appending.
- Use `detect_header_row` from the worked example.

In [9]:
# TODO: def load_all_sheets(path): ...
# df_all = load_all_sheets("raw_transactions.xlsx")

## 1.2 Encoding & corrupted-file defense

**Concept:** CSV files from legacy systems arrive in `latin1`, `cp1252`, or `iso-8859-1`
encoding, not `utf-8`. Forcing `utf-8` produces `UnicodeDecodeError` or, worse, silently
mangles accented characters. The defense: try `utf-8` first, catch the error, fall back to
`latin1`. For truly broken files, `on_bad_lines="warn"` (pandas >= 1.3) skips malformed lines
instead of crashing the whole read.

**FAANG context:** At Amazon, inbound vendor files arrive in every encoding imaginable.
Production pipelines wrap every `read_csv` in an encoding-fallback chain and log which encoding
succeeded, so the team knows when a vendor changes their export settings.

In [10]:
import pandas as pd

def safe_read_csv(path, **kwargs):
    # Production-grade CSV reader with encoding fallback and bad-line defense.
    encodings = ["utf-8", "utf-8-sig", "latin1", "cp1252", "iso-8859-1"]
    for enc in encodings:
        try:
            df = pd.read_csv(path, encoding=enc, on_bad_lines="warn", **kwargs)
            print(f"  Loaded {path} with encoding={enc}, shape={df.shape}")
            return df
        except (UnicodeDecodeError, UnicodeError):
            continue
    raise RuntimeError(f"Could not read {path} with any of {encodings}")

df_cust = safe_read_csv("customer_master.csv")
df_cust.head(3)

  Loaded customer_master.csv with encoding=utf-8, shape=(200, 5)


,CustID,CompanyName,Segment,Country,CreditLimit
0,CUST-0001,Acme Corp,Government,KE,188440.62
1,CUST-0002,Globex International,Government,US,305801.54
2,CUST-0003,Initech Solutions,Startup,RW,112588.38


## 1.3 Memory-safe reading: `usecols` and chunked loading

**Concept:** Loading every column when you only need three wastes RAM and time. `usecols=[...]`
drops unwanted columns *at read time*, before they ever touch memory. For files too large to fit
in RAM at all, `chunksize=N` returns an iterator of DataFrames, each with N rows, that you can
process and discard one at a time.

**FAANG context:** A 20 GB log file that fits comfortably in Spark will OOM-kill a pandas process.
Chunking lets you aggregate or filter in pandas without needing a cluster.

In [11]:
# usecols: read only the columns you need
df_slim = pd.read_excel(
    "raw_transactions.xlsx", sheet_name="Apr_Jun",
    usecols=["Product_Code", "Qty", " Amount (USD)"],
)
print(f"Slim shape (3 cols only): {df_slim.shape}")
print(f"Slim memory: {df_slim.memory_usage(deep=True).sum() / 1024:.1f} KB")

# Chunked reading: process a CSV in N-row pieces without loading everything at once
total_rows = 0
total_amount = 0.0
for chunk in pd.read_csv("customer_master.csv", chunksize=50):
    total_rows += len(chunk)
    total_amount += chunk["CreditLimit"].sum()
print(f"\nChunked read: {total_rows} rows, total CreditLimit = {total_amount:,.2f}")

Slim shape (3 cols only): (1000, 3)
Slim memory: 37.9 KB

Chunked read: 200 rows, total CreditLimit = 48,311,064.27


### 🧩 Practice 1.2
Using the `safe_read_csv` function, load `customer_master.csv` but read **only** the `CustID`
and `Country` columns (via `usecols`). Store the resulting DataFrame in `df_id_country`.
Confirm it has exactly 2 columns.

**Hints:** `safe_read_csv("customer_master.csv", usecols=["CustID", "Country"])`

In [12]:
# TODO: df_id_country = safe_read_csv(...)

---
# Module 2: Full-Spectrum Data Cleaning & Type Enforcement

Cleaning is where most real-world analyst time goes. This module covers every common pattern:
normalizing column names and text, parsing messy dates, casting types for memory efficiency,
and the critical practice of *quarantining* bad records instead of silently dropping them.

## 2.1 Column-name normalization

**Concept:** Strip whitespace, lowercase, replace spaces/special characters with underscores.
Do this *first*, before any other cleaning, so every subsequent line of code uses predictable,
typo-proof column names. This is a one-liner with `.str` accessor methods.

**FAANG context:** Column-name normalization is the literal first line of every transform at
Google's BigQuery ingestion layer. It prevents downstream SQL and pandas code from silently
failing because of an invisible trailing space.

In [13]:
import re

def normalize_columns(df):
    # Production column normalizer: strip, lowercase, collapse special chars to underscore.
    df = df.copy()
    df.columns = (
        df.columns
          .str.strip()
          .str.lower()
          .str.replace(r"[^a-z0-9]+", "_", regex=True)  # any non-alphanumeric -> underscore
          .str.strip("_")                                  # remove leading/trailing underscores
    )
    return df

df_txn = normalize_columns(df_stacked)
print(df_txn.columns.tolist())
df_txn.head(3)

['txn_id', 'order_date', 'customer_id', 'company_name', 'product_code', 'qty', 'amount_usd', 'region', 'payment_method', 'status', 'sales_rep', 'notes']


,txn_id,order_date,customer_id,company_name,product_code,qty,amount_usd,region,payment_method,status,sales_rep,notes
0,TXN-100000,18-Mar-2026,CUST-0197,Abstergo Industries,SKU-N192,46.0,2998.55,Central,Wire Transfer,Cancelled,Dan,Rush
1,TXN-100002,16/01/2026,CUST-0103,Weyland-Yutani,SKU-U820,18.0,109.26,Central,PayPal,Cancelled,Dan,Bulk discount
2,TXN-100004,2026-01-26,CUST-0082,Oscorp Technologies,SKU-D560,7.0,1952.25,South,Wire Transfer,Cancelled,Eve,NaN


## 2.2 String sanitization & fuzzy matching

**Concept:** `.str.strip()` and `.str.title()` handle whitespace and casing. But what about
misspellings? `"Acme Corp"` vs `"acme  corp"` vs `"ACME CORP "` are all the same company.
`rapidfuzz.process.extractOne()` finds the closest match from a master list using Levenshtein
distance, with a configurable score threshold (80+ is usually safe for company names).

**FAANG context:** At Meta, entity resolution (mapping messy user-entered names to canonical
records) runs fuzzy matching at scale. The key insight: do deterministic cleaning first
(strip/lower/deduplicate spaces), then only run fuzzy matching on the remaining mismatches.
This is 10x faster than fuzzing everything.

In [14]:
from rapidfuzz import process, fuzz

# Step 1: deterministic cleaning (fast, handles 80%+ of cases)
df_txn["company_name"] = (
    df_txn["company_name"]
      .str.strip()
      .str.lower()
      .str.replace(r"\s+", " ", regex=True)  # collapse multiple spaces to one
      .str.title()                              # "acme corp" -> "Acme Corp"
)

# Step 2: fuzzy matching against the canonical company list (handles misspellings)
canonical_companies = [
    "Acme Corp", "Globex International", "Initech Solutions", "Umbrella Holdings",
    "Stark Industries", "Wayne Enterprises", "Soylent Corp", "Cyberdyne Systems",
    "Oscorp Technologies", "Aperture Science", "Hooli Inc", "Pied Piper",
    "Massive Dynamic", "Tyrell Corporation", "Weyland-Yutani", "Abstergo Industries",
    "Vought International", "Lexcorp", "Dunder Mifflin", "Prestige Worldwide",
]

def fuzzy_match_company(name, choices=canonical_companies, threshold=80):
    if pd.isna(name):
        return name
    result = process.extractOne(name, choices, scorer=fuzz.ratio, score_cutoff=threshold)
    return result[0] if result else name  # return canonical match, or original if no match

# Apply fuzzy matching to the unique values first (much faster than row-by-row)
unique_names = df_txn["company_name"].dropna().unique()
fuzzy_map = {name: fuzzy_match_company(name) for name in unique_names}
df_txn["company_name_clean"] = df_txn["company_name"].map(fuzzy_map)

# Show how many names were corrected
changed = (df_txn["company_name"] != df_txn["company_name_clean"]).sum()
print(f"Fuzzy matching corrected {changed} out of {len(df_txn)} company name entries.")
print(f"Unique companies before: {df_txn['company_name'].nunique()}")
print(f"Unique companies after:  {df_txn['company_name_clean'].nunique()}")

Fuzzy matching corrected 0 out of 2000 company name entries.
Unique companies before: 20
Unique companies after:  20


### 🧩 Practice 2.2
The `region` column still has casing and whitespace variants. Clean it using deterministic
methods only (no fuzzy matching needed — the canonical set is just 5 values). Store the number
of distinct `region` values *before* and *after* cleaning in `region_before` and `region_after`.

**Hints:**
- `.str.strip().str.title()` should be sufficient.
- `df_txn["region"].nunique()` counts distinct values.

In [15]:
# TODO: clean df_txn["region"], compute region_before and region_after

## 2.3 Datetime normalization

**Concept:** `pd.to_datetime(series, errors="coerce")` handles mixed formats automatically.
`errors="coerce"` turns unparseable entries into `NaT` instead of crashing. After parsing,
the `.dt` accessor gives you derived fields like month, weekday, quarter, and business-day
differences (`np.busday_count`).

**FAANG context:** Date columns are the #1 source of silent data loss in ETL pipelines.
The "coerce + count NaT" pattern catches exactly how many rows failed, so you can decide
whether to fix, impute, or quarantine them.

In [16]:
df_txn["order_date"] = pd.to_datetime(df_txn["order_date"], errors="coerce")
n_failed = df_txn["order_date"].isna().sum()
print(f"Date parsing failures: {n_failed} ({n_failed/len(df_txn)*100:.1f}%)")

# Derived date fields -- essentially free once the column is a real datetime
df_txn["order_month"] = df_txn["order_date"].dt.to_period("M").astype(str)
df_txn["order_quarter"] = df_txn["order_date"].dt.quarter
df_txn["order_weekday"] = df_txn["order_date"].dt.day_name()

# Business-day count from the start of the year (useful for finance reporting)
year_start = pd.Timestamp("2026-01-01")
df_txn["biz_days_from_start"] = df_txn["order_date"].apply(
    lambda d: np.busday_count(year_start.date(), d.date()) if pd.notna(d) else np.nan
)

df_txn[["order_date", "order_month", "order_quarter", "order_weekday", "biz_days_from_start"]].head()

Date parsing failures: 1508 (75.4%)


,order_date,order_month,order_quarter,order_weekday,biz_days_from_start
0,2026-03-18,2026-03,1.0,Wednesday,54.0
1,NaT,NaN,NaN,NaN,NaN
2,NaT,NaN,NaN,NaN,NaN
3,NaT,NaN,NaN,NaN,NaN
4,2026-03-23,2026-03,1.0,Monday,57.0


### 🧩 Practice 2.3
Create a column `is_month_end` that is `True` if the order date falls on the last business day
of its month, `False` otherwise (and `NaN` for missing dates).

**Hints:**
- `pd.tseries.offsets.BMonthEnd(0)` snaps a date *forward* to the current month's last business
  day (or leaves it unchanged if it already is).
- Compare `df_txn["order_date"]` to
  `df_txn["order_date"] + pd.tseries.offsets.BMonthEnd(0)`.

In [17]:
# TODO: create df_txn["is_month_end"]

## 2.4 Type casting & memory optimization

**Concept:** pandas defaults to `float64` and `object` for almost everything. Downcasting
numeric columns (`float64` -> `float32`, or `int64` -> `int16` when the range fits) can halve
memory usage. Converting low-cardinality string columns to `category` dtype can reduce memory
by 90%+ and speed up groupbys.

**FAANG context:** In Google's internal pandas-like libraries, dtype optimization is automatic
on read. In open-source pandas you have to do it yourself, but the payoff on a 10M-row DataFrame
is the difference between fitting in RAM and getting OOM-killed.

In [18]:
def optimize_dtypes(df):
    # Production dtype optimizer: downcast numerics and categorize low-cardinality strings.
    df = df.copy()
    mem_before = df.memory_usage(deep=True).sum()

    for col in df.select_dtypes(include=["float64"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="float")
    for col in df.select_dtypes(include=["int64", "int32"]).columns:
        df[col] = pd.to_numeric(df[col], downcast="integer")
    for col in df.select_dtypes(include=["object"]).columns:
        if df[col].nunique() / max(len(df), 1) < 0.05:  # <5% cardinality -> category
            df[col] = df[col].astype("category")

    mem_after = df.memory_usage(deep=True).sum()
    savings_pct = (1 - mem_after / mem_before) * 100
    print(f"Memory: {mem_before/1024**2:.2f} MB -> {mem_after/1024**2:.2f} MB ({savings_pct:.1f}% savings)")
    return df

# First cast qty and amount to numeric (they were object due to hidden nulls)
df_txn["qty"] = pd.to_numeric(df_txn["qty"], errors="coerce")
df_txn["amount_usd"] = pd.to_numeric(df_txn["amount_usd"], errors="coerce")

# Fill missing qty with 1 (business default: minimum order)
df_txn["qty"] = df_txn["qty"].fillna(1).astype(int)

df_txn = optimize_dtypes(df_txn)

Memory: 0.43 MB -> 0.14 MB (68.5% savings)


/tmp/ipykernel_590/2343825620.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for col in df.select_dtypes(include=["object"]).columns:


## 2.5 Outlier detection & quarantine

**Concept:** Rather than silently dropping outliers (which loses data and breaks audit trails),
production pipelines **quarantine** them: separate the flagged rows into a dedicated DataFrame
that gets written to a `_quarantine` sheet or table for human review, while the main pipeline
continues with only validated records.

Two standard detection methods:
- **IQR method:** Flag any value below Q1 - 1.5*IQR or above Q3 + 1.5*IQR.
- **Z-score method:** Flag any value with |z| > 3 (more than 3 standard deviations from mean).

**FAANG context:** At Amazon, every data pipeline has a quarantine table. The business rule is:
never silently drop a record. Quarantined records get reviewed weekly and either corrected or
confirmed as genuine edge cases.

In [19]:
def quarantine_outliers(df, col, method="iqr", threshold=1.5, z_thresh=3):
    # Split df into (valid_df, quarantine_df) based on outlier detection on col.
    s = df[col].dropna()
    if method == "iqr":
        q1, q3 = s.quantile(0.25), s.quantile(0.75)
        iqr = q3 - q1
        lower, upper = q1 - threshold * iqr, q3 + threshold * iqr
        outlier_mask = (df[col] < lower) | (df[col] > upper)
        print(f"IQR bounds: [{lower:.2f}, {upper:.2f}]")
    elif method == "zscore":
        mean, std = s.mean(), s.std()
        z = (df[col] - mean) / std
        outlier_mask = z.abs() > z_thresh
    else:
        raise ValueError("method must be 'iqr' or 'zscore'")

    quarantined = df[outlier_mask].copy()
    quarantined["_quarantine_reason"] = f"Outlier in {col} ({method})"
    valid = df[~outlier_mask].copy()
    print(f"Quarantined {len(quarantined)} rows ({len(quarantined)/len(df)*100:.1f}%)")
    return valid, quarantined

df_valid, df_quarantine = quarantine_outliers(df_txn, "amount_usd", method="iqr")
print(f"\nValid: {len(df_valid)} rows  |  Quarantine: {len(df_quarantine)} rows")
df_quarantine[["txn_id", "amount_usd", "_quarantine_reason"]].head()

IQR bounds: [-3530.66, 9726.63]
Quarantined 39 rows (1.9%)

Valid: 1961 rows  |  Quarantine: 39 rows


,txn_id,amount_usd,_quarantine_reason
69,TXN-100138,65070.28,Outlier in amount_usd (iqr)
83,TXN-100166,690455.99,Outlier in amount_usd (iqr)
176,TXN-100352,296257.52,Outlier in amount_usd (iqr)
252,TXN-100504,45495.30,Outlier in amount_usd (iqr)
327,TXN-100654,191931.30,Outlier in amount_usd (iqr)


### 🧩 Practice 2.5
Run the quarantine using the **z-score** method (with default `z_thresh=3`) on the same
`amount_usd` column. Store the resulting valid and quarantine DataFrames in `df_valid_z` and
`df_quarantine_z`. Compare how many rows each method quarantines.

**Hints:**
- `quarantine_outliers(df_txn, "amount_usd", method="zscore")`
- IQR typically catches more outliers than z-score for skewed distributions.

In [20]:
# TODO: df_valid_z, df_quarantine_z = quarantine_outliers(...)

### ✅ Module 2 Checkpoint
Run this cell to verify the cleaned DataFrame is in good shape.

In [21]:
assert "txn_id" in df_valid.columns, "Column names should be normalized by now"
assert df_valid["order_date"].dtype.kind == "M", "order_date should be datetime"
assert df_valid["qty"].dtype.kind in "iu", "qty should be integer"
assert df_valid["amount_usd"].dtype.kind == "f", "amount_usd should be float"
print(f"Module 2 checkpoint passed. Valid rows: {len(df_valid):,}, Quarantined: {len(df_quarantine):,}")

Module 2 checkpoint passed. Valid rows: 1,961, Quarantined: 39


---
# Module 3: Advanced Transformations & Relational Analytics

With clean, typed data in hand, this module covers the analytical power moves: merging reference
tables (the VLOOKUP/XLOOKUP equivalent), multi-key joins with collision handling, named
aggregations, and window functions.

## 3.1 VLOOKUP / XLOOKUP equivalents with `pd.merge`

**Concept:** `pd.merge(left, right, on="key", how="left")` is the pandas equivalent of
VLOOKUP/XLOOKUP for an *entire table at once*. Key considerations:

- **`how="left"`** keeps every row from the left table (like XLOOKUP with `if_not_found`).
- **`suffixes=`** prevents silent column-name collisions when both tables share a name.
- **`validate="m:1"`** asserts the relationship cardinality — if a "many-to-one" join
  accidentally becomes "many-to-many" (because the lookup table has duplicate keys), it raises
  an error instead of silently multiplying your rows.

**FAANG context:** At Google, every join in a pipeline specifies `validate=` explicitly. Silent
row multiplication from a bad join key is one of the most expensive bugs in analytics — your
numbers are all wrong, but plausibly so, and nobody notices until the CFO asks why revenue is 2x.

In [22]:
import pandas as pd
import json

# Load reference tables
df_cust = pd.read_csv("customer_master.csv")
with open("product_catalog.json") as f:
    df_prod = pd.DataFrame(json.load(f))

# Normalize column names to match df_valid
df_cust.columns = df_cust.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")
df_prod.columns = df_prod.columns.str.strip().str.lower().str.replace(r"[^a-z0-9]+", "_", regex=True).str.strip("_")

# Rename to match the transaction table keys
df_cust = df_cust.rename(columns={"custid": "customer_id"})
df_prod = df_prod.rename(columns={"productcode": "product_code"})

# LEFT JOIN: bring product details into transactions (like a VLOOKUP on Product_Code)
# validate="m:1" ensures we don't accidentally get duplicate keys in the product catalog
df_enriched = df_valid.merge(
    df_prod, on="product_code", how="left",
    validate="m:1",    # assert: each product_code appears at most once in df_prod
    suffixes=("", "_product"),  # if both tables have a "category" column, the product's gets a suffix
)

# LEFT JOIN: bring customer details into the enriched table
df_enriched = df_enriched.merge(
    df_cust, on="customer_id", how="left",
    validate="m:1",
    suffixes=("", "_customer"),
)

print(f"Enriched shape: {df_enriched.shape}")
unmatched_prod = df_enriched["productname"].isna().sum()
unmatched_cust = df_enriched["companyname"].isna().sum()
print(f"Unmatched products: {unmatched_prod}  |  Unmatched customers: {unmatched_cust}")

# Calculated column: line-item revenue
df_enriched["line_revenue"] = df_enriched["qty"] * df_enriched["unitprice"]
df_enriched.head()

Enriched shape: (1961, 25)
Unmatched products: 0  |  Unmatched customers: 0


,txn_id,order_date,customer_id,company_name,product_code,qty,amount_usd,region,payment_method,status,...,biz_days_from_start,productname,category,unitprice,weight_kg,companyname,segment,country,creditlimit,line_revenue
0,TXN-100000,2026-03-18,CUST-0197,Abstergo Industries,SKU-N192,46,2998.55,Central,Wire Transfer,Cancelled,...,54.0,Widget 192,Electronics,170.28,1.0,Vought International,Government,NG,245882.59,7832.88
1,TXN-100002,NaT,CUST-0103,Weyland-Yutani,SKU-U820,18,109.26,Central,PayPal,Cancelled,...,NaN,Widget 820,Furniture,508.92,9.6,Initech Solutions,Government,BR,40904.48,9160.56
2,TXN-100004,NaT,CUST-0082,Oscorp Technologies,SKU-D560,7,1952.25,South,Wire Transfer,Cancelled,...,NaN,Widget 560,Furniture,1628.79,37.2,Globex International,Enterprise,NG,241069.15,11401.53
3,TXN-100006,NaT,CUST-0001,Initech Solutions,SKU-D560,6,1731.90,West,PayPal,Refunded,...,NaN,Widget 560,Furniture,1628.79,37.2,Acme Corp,Government,KE,188440.62,9772.74
4,TXN-100008,2026-03-23,CUST-0154,Pied Piper,SKU-Y891,19,722.53,South,PayPal,Refunded,...,57.0,Widget 891,Electronics,859.40,15.9,Tyrell Corporation,Enterprise,KE,144129.37,16328.60


### 🧩 Practice 3.1
Perform an **inner join** (only keep rows that match on both sides) of `df_valid` and `df_prod`
on `product_code`. Store the result in `df_inner` and compare its row count to `df_enriched`
(which used a left join). The difference tells you how many transaction rows have a product code
that doesn't exist in the catalog.

**Hints:**
- `how="inner"` drops non-matching rows from *both* sides.
- `len(df_enriched) - len(df_inner)` = unmatched product rows.

In [23]:
# TODO: df_inner = df_valid.merge(df_prod, on="product_code", how="inner", ...)

## 3.2 Named aggregations & custom functions

**Concept:** `df.groupby(...).agg(new_col=("source_col", "func"))` (named aggregation) gives
each output column a clean name directly, avoiding the MultiIndex headache that plain `.agg()`
produces. You can also pass custom functions (lambdas or named functions) for aggregations that
go beyond `sum`/`mean`/`count`.

**FAANG context:** Named aggregations are the standard pattern in production analytics code at
Meta and Google because they produce flat, self-documenting column names that downstream SQL
and dashboards can consume without post-processing.

In [24]:
# Named aggregation: clean, flat output -- no MultiIndex to wrestle with
rep_summary = df_enriched.groupby("sales_rep").agg(
    total_revenue=("line_revenue", "sum"),
    order_count=("txn_id", "count"),
    avg_order_value=("line_revenue", "mean"),
    unique_customers=("customer_id", "nunique"),
    max_single_order=("line_revenue", "max"),
).round(2).sort_values("total_revenue", ascending=False)

print("Sales Rep Summary:")
rep_summary

Sales Rep Summary:


,total_revenue,order_count,avg_order_value,unique_customers,max_single_order
sales_rep,,,,,
Dan,5965654.87,287,20786.25,154,86088.50
Alice,5946013.91,259,22957.58,144,98360.50
Carol,5773923.69,274,21072.71,147,98360.50
Frank,5627145.41,252,22329.94,145,92458.87
Eve,5598657.52,268,20890.51,154,96393.29
Bob,5420226.31,273,19854.31,156,86088.50
Alice,267252.99,15,17816.87,15,63713.25
Bob,231160.04,10,23116.00,10,46145.76
Eve,224228.93,14,16016.35,14,61983.72


### 🧩 Practice 3.2
Build a summary called `category_region_summary` grouped by **both** `category` and `region`,
showing: `total_qty` (sum of qty), `total_revenue` (sum of line_revenue), and `avg_price`
(mean of unitprice). Sort by `total_revenue` descending.

**Hints:**
- `.groupby(["category", "region"]).agg(...)` groups by two columns at once.
- Use the same named-aggregation syntax as the worked example.

In [25]:
# TODO: category_region_summary = ...

## 3.3 Window functions: `transform`, `rolling`, `expanding`

**Concept:** `.transform()` applies a function to each group but returns a Series **the same
length as the original DataFrame** — perfect for adding a "group total" or "percentage of group"
column without collapsing rows. `.rolling(N)` computes moving windows (e.g., 7-day moving
average), and `.expanding()` computes cumulative statistics.

**FAANG context:** At Amazon, rolling averages and cumulative sums over time are the bread and
butter of operational dashboards. `transform` is the go-to for computing "this row's share of
its group's total" without a self-join — a pattern that appears in almost every finance report.

In [26]:
# transform: "what % of the region's total revenue is each order?"
df_enriched["region_total_rev"] = df_enriched.groupby("region")["line_revenue"].transform("sum")
df_enriched["pct_of_region"] = (df_enriched["line_revenue"] / df_enriched["region_total_rev"] * 100).round(2)

# rolling: 7-period moving average of line_revenue (sorted by date first)
df_sorted = df_enriched.dropna(subset=["order_date"]).sort_values("order_date")
df_sorted["rolling_7_avg"] = df_sorted["line_revenue"].rolling(window=7, min_periods=1).mean().round(2)

# expanding: cumulative sum of revenue over time
df_sorted["cumulative_revenue"] = df_sorted["line_revenue"].expanding().sum().round(2)

df_sorted[["order_date", "line_revenue", "rolling_7_avg", "cumulative_revenue"]].head(10)

,order_date,line_revenue,rolling_7_avg,cumulative_revenue
901,2026-01-01,4581.04,4581.04,4581.04
880,2026-01-01,6515.16,5548.10,11096.20
209,2026-01-02,1750.47,4282.22,12846.67
108,2026-01-03,8525.40,5343.02,21372.07
333,2026-01-03,170.28,4308.47,21542.35
627,2026-01-03,3371.35,4152.28,24913.70
182,2026-01-04,48068.50,10426.03,72982.20
436,2026-01-04,9926.54,11189.67,82908.74
820,2026-01-04,11705.16,11931.10,94613.90
592,2026-01-04,24456.00,15174.75,119069.90


### 🧩 Practice 3.3
Using `transform`, add a column `rep_order_rank` that ranks each order *within its sales rep*
by `line_revenue` (highest = rank 1). This is the pandas equivalent of SQL's
`RANK() OVER (PARTITION BY sales_rep ORDER BY line_revenue DESC)`.

**Hints:**
- `.groupby("sales_rep")["line_revenue"].rank(method="dense", ascending=False)`
- `.rank()` with `transform` returns a same-length Series.

In [27]:
# TODO: df_enriched["rep_order_rank"] = ...

## 3.4 Pivot tables & multi-index flattening

**Concept:** `pd.pivot_table()` recreates Excel's PivotTable. The resulting MultiIndex columns
(when you use `columns=`) are hard to work with in downstream code, so production pipelines
flatten them immediately using a `_`-joined string of the level names.

**FAANG context:** Flattening MultiIndex columns is a mandatory post-processing step in any
pipeline that writes results to a database or dashboard — neither SQL tables nor BI tools
support hierarchical column names.

In [28]:
pivot = pd.pivot_table(
    df_enriched,
    values="line_revenue",
    index="region",
    columns="category",
    aggfunc=["sum", "mean"],
    fill_value=0,
    margins=True,
    margins_name="Grand Total",
)

print("Raw pivot (MultiIndex columns):")
print(pivot.columns.tolist()[:6], "...")

# Flatten the MultiIndex columns: ("sum", "Electronics") -> "sum_Electronics"
pivot.columns = ["_".join(str(c) for c in col).strip("_") for col in pivot.columns.values]
print("\nFlattened columns:", pivot.columns.tolist()[:6], "...")
pivot

Raw pivot (MultiIndex columns):
[('sum', 'Electronics'), ('sum', 'Furniture'), ('sum', 'Office Supplies'), ('sum', 'Services'), ('sum', 'Software'), ('sum', 'Grand Total')] ...

Flattened columns: ['sum_Electronics', 'sum_Furniture', 'sum_Office Supplies', 'sum_Services', 'sum_Software', 'sum_Grand Total'] ...


,sum_Electronics,sum_Furniture,sum_Office Supplies,sum_Services,sum_Software,sum_Grand Total,mean_Electronics,mean_Furniture,mean_Office Supplies,mean_Services,mean_Software,mean_Grand Total
region,,,,,,,,,,,,
Central,0.00,40084.10,83377.48,62179.86,137704.70,323346.14,0.000000,40084.100000,13896.246667,12435.972000,68852.350000,23096.152857
East,47008.93,89274.90,35625.19,30321.50,25043.20,227273.72,47008.930000,29758.300000,17812.595000,15160.750000,25043.200000,25252.635556
North,0.00,13191.20,3217.27,9140.85,66453.52,92002.84,0.000000,13191.200000,1608.635000,4570.425000,22151.173333,11500.355000
South,17737.83,99037.82,0.00,32740.60,85449.44,234965.69,17737.830000,33012.606667,0.000000,10913.533333,28483.146667,23496.569000
West,4257.00,0.00,26970.80,3922.32,34080.42,69230.54,4257.000000,0.000000,26970.800000,3922.320000,17040.210000,13846.108000
CENTRAL,47548.58,30923.20,13841.94,0.00,40477.10,132790.82,15849.526667,30923.200000,6920.970000,0.000000,13492.366667,14754.535556
Central,681996.22,1552513.22,2521343.43,1180266.01,1933594.20,7869713.08,13918.290204,25875.220333,23131.591101,14050.785833,33922.705263,21921.206351
Central,0.00,0.00,3311.94,6490.43,82377.38,92179.75,0.000000,0.000000,3311.940000,6490.430000,41188.690000,23044.937500
EAST,0.00,74082.44,113860.99,21465.84,90232.13,299641.40,0.000000,24694.146667,37953.663333,10732.920000,30077.376667,27240.127273


### 🧩 Practice 3.4
Build a pivot table of **order count** (aggfunc="count") with `order_month` as rows and
`status` as columns. Flatten the column names and store the result in `pivot_month_status`.

**Hints:**
- `values="txn_id"` with `aggfunc="count"` gives order count.
- When there's only one aggfunc, the MultiIndex may be simpler — but flatten it anyway for
  robustness.

In [29]:
# TODO: pivot_month_status = ...

---
# Module 4: Big Tech Data Quality & Pipeline Architecture

This module covers the engineering patterns that separate a "script" from a "pipeline": schema
assertions that catch data drift before it propagates, vectorized logic that replaces slow loops,
and DuckDB for SQL-speed queries on pandas DataFrames.

## 4.1 Schema enforcement & assertions

**Concept:** Programmatic assertions (`assert`, or structured checks) verify *invariants* that
your data must satisfy before downstream code runs: non-null constraints, unique key checks,
value-range bounds, and referential integrity. These are the pandas equivalent of database
constraints / Great Expectations suites.

**FAANG context:** At Google, every production pipeline has a "data contract" — a set of
assertions that must pass before data flows to the next stage. If any assertion fails, the
pipeline halts and pages the on-call engineer rather than silently writing bad data to the
warehouse.

In [30]:
def run_data_quality_checks(df, checks):
    # Run a list of data quality checks and report pass/fail for each.
    # Each check is a dict: {"name": str, "check": callable(df) -> bool, "severity": "error"|"warn"}
    results = []
    all_passed = True
    for chk in checks:
        try:
            passed = chk["check"](df)
        except Exception as e:
            passed = False
        status = "PASS" if passed else ("FAIL" if chk.get("severity") == "error" else "WARN")
        if not passed:
            all_passed = False
        results.append({"check": chk["name"], "status": status, "severity": chk.get("severity", "error")})
        icon = "+" if passed else "X"
        print(f"  [{icon}] {chk['name']} -> {status}")
    return all_passed, pd.DataFrame(results)

quality_checks = [
    {"name": "txn_id is unique",
     "check": lambda df: df["txn_id"].is_unique,
     "severity": "error"},
    {"name": "order_date has no nulls",
     "check": lambda df: df["order_date"].notna().all(),
     "severity": "warn"},
    {"name": "qty is positive",
     "check": lambda df: (df["qty"] > 0).all(),
     "severity": "error"},
    {"name": "amount_usd is non-negative",
     "check": lambda df: (df["amount_usd"].dropna() >= 0).all(),
     "severity": "error"},
    {"name": "region in valid set",
     "check": lambda df: set(df["region"].dropna().unique()) <= {"East", "West", "North", "South", "Central"},
     "severity": "warn"},
    {"name": "row count > 100",
     "check": lambda df: len(df) > 100,
     "severity": "error"},
]

print("Data Quality Report:")
all_ok, dq_report = run_data_quality_checks(df_valid, quality_checks)
print(f"\nOverall: {'ALL PASSED' if all_ok else 'ISSUES DETECTED'}")

Data Quality Report:
  [X] txn_id is unique -> FAIL
  [X] order_date has no nulls -> WARN
  [+] qty is positive -> PASS
  [+] amount_usd is non-negative -> PASS
  [X] region in valid set -> WARN
  [+] row count > 100 -> PASS

Overall: ISSUES DETECTED


### 🧩 Practice 4.1
Add two more checks to the `quality_checks` list: (1) `"status values are valid"` — ensure every
`status` value is in `{"Completed", "Pending", "Refunded", "Cancelled"}`, and (2)
`"customer_id format check"` — ensure every `customer_id` matches the regex pattern
`^CUST-\d{4}$`. Re-run `run_data_quality_checks` with the expanded list.

**Hints:**
- `df["col"].str.match(r"^CUST-\d{4}$").all()` checks a regex pattern across a column.
- Remember to handle NaN values — `.dropna()` before `.str.match()`.

In [31]:
# TODO: add two more checks and re-run

## 4.2 Vectorized logic over iteration

**Concept:** Python `for` loops over DataFrame rows run at ~1000x slower than vectorized
operations. `.apply()` is slightly better but still calls a Python function per row.
`np.where(cond, if_true, if_false)` and `np.select(conditions, choices, default)` are the
vectorized equivalents of Excel's `IF` and nested `IF` — they run at C speed.

**FAANG context:** Replacing a single `.apply()` call with `np.select()` reduced a Meta
pipeline's transform step from 45 minutes to 8 seconds on a 50M-row table. This is the #1
performance optimization in pandas code.

In [32]:
import numpy as np
import time

# Scenario: classify orders into tiers based on line_revenue
# BAD: .apply() with a Python function (slow, not vectorized)
def classify_slow(row):
    if row["line_revenue"] >= 5000: return "Platinum"
    elif row["line_revenue"] >= 1000: return "Gold"
    elif row["line_revenue"] >= 200: return "Silver"
    else: return "Bronze"

t0 = time.time()
df_enriched["tier_slow"] = df_enriched.apply(classify_slow, axis=1)
time_slow = time.time() - t0

# GOOD: np.select (vectorized, runs at C speed)
conditions = [
    df_enriched["line_revenue"] >= 5000,
    df_enriched["line_revenue"] >= 1000,
    df_enriched["line_revenue"] >= 200,
]
choices = ["Platinum", "Gold", "Silver"]

t0 = time.time()
df_enriched["tier_fast"] = np.select(conditions, choices, default="Bronze")
time_fast = time.time() - t0

print(f".apply() time: {time_slow:.4f}s")
print(f"np.select time: {time_fast:.4f}s")
print(f"Speedup: {time_slow / max(time_fast, 0.0001):.0f}x")
assert (df_enriched["tier_slow"] == df_enriched["tier_fast"]).all(), "Results should be identical"

.apply() time: 0.0115s
np.select time: 0.0007s
Speedup: 16x


### 🧩 Practice 4.2
Using `np.select`, create a column `amount_bucket` that classifies `amount_usd` into four
buckets: `"Micro"` (< 50), `"Small"` (50-500), `"Medium"` (500-5000), and `"Large"` (>= 5000).
Do NOT use `.apply()`.

**Hints:**
- Define 3 conditions (the 4th is the default).
- `np.select(conditions, choices, default="Large")`

In [33]:
# TODO: df_enriched["amount_bucket"] = np.select(...)

## 4.3 DuckDB interoperability

**Concept:** DuckDB lets you run SQL queries **directly on pandas DataFrames** without any
import/export step — it reads the in-memory DataFrame as a virtual table. For analytical queries
(aggregations, window functions, joins) on 100K+ rows, DuckDB is often 10-50x faster than
equivalent pandas code because it uses a columnar vectorized execution engine.

**FAANG context:** DuckDB has become the "Swiss army knife" for data engineers who need SQL
speed without a server. It's used at scale for ad-hoc analysis, data quality checks on large
files, and as a fast alternative to Spark for single-node workloads.

In [34]:
import duckdb
import time

# Load the 500k-row Parquet file
df_big = pd.read_parquet("big_events.parquet")
print(f"Loaded big_events: {len(df_big):,} rows, {df_big.memory_usage(deep=True).sum()/1024**2:.1f} MB")

# --- pandas aggregation ---
t0 = time.time()
pandas_result = (
    df_big
    .groupby("event_type")
    .agg(total_amount=("amount", "sum"), event_count=("event_id", "count"),
         avg_amount=("amount", "mean"))
    .round(2)
)
time_pandas = time.time() - t0

# --- DuckDB: SQL directly on the pandas DataFrame ---
t0 = time.time()
duckdb_result = duckdb.sql('''
    SELECT event_type,
           ROUND(SUM(amount), 2) AS total_amount,
           COUNT(event_id) AS event_count,
           ROUND(AVG(amount), 2) AS avg_amount
    FROM df_big
    GROUP BY event_type
    ORDER BY total_amount DESC
''').df()
time_duckdb = time.time() - t0

print(f"\npandas time: {time_pandas:.3f}s")
print(f"DuckDB time: {time_duckdb:.3f}s")
print(f"Speedup: {time_pandas / max(time_duckdb, 0.001):.1f}x")
duckdb_result

Loaded big_events: 500,000 rows, 21.8 MB



pandas time: 0.025s
DuckDB time: 0.118s
Speedup: 0.2x


,event_type,total_amount,event_count,avg_amount
0,logout,5029704.38,100100,50.25
1,view,5004734.03,99887,50.10
2,signup,5003375.29,100087,49.99
3,purchase,5000204.71,99888,50.06
4,click,4972264.52,100038,49.70


### 🧩 Practice 4.3
Using DuckDB, write a SQL query on `df_big` that computes a **window function**: for each row,
calculate the running total of `amount` within each `user_id`, ordered by `ts`. Return the first
10 rows. Store the result in `df_window_result`.

**Hints:**
- `SUM(amount) OVER (PARTITION BY user_id ORDER BY ts) AS running_total`
- `duckdb.sql("SELECT ..., SUM(amount) OVER (...) ... FROM df_big LIMIT 10").df()`

In [35]:
# TODO: df_window_result = duckdb.sql(...)

---
# Module 5: Excel Automation, Formatting & Multi-Tab Exporting

The last mile: getting your clean, validated, enriched data back into a polished `.xlsx` workbook
that a non-technical stakeholder can open and immediately understand — multiple sheets, styled
headers, currency formatting, zebra striping, conditional formatting, live formulas, and embedded
charts, all generated automatically.

## 5.1 Multi-sheet writing with ExcelWriter

**Concept:** `pd.ExcelWriter(path, engine="openpyxl")` in a `with` block lets you write multiple
DataFrames to different sheets of the same workbook. The file is finalized only when the block
exits, ensuring atomicity. Adding `mode="a"` and `if_sheet_exists="replace"` lets you add sheets
to an existing workbook without overwriting the others.

**FAANG context:** Production reports almost always have multiple tabs: raw data, summary,
quarantine log, metadata. Writing them all atomically in one `with` block prevents the failure
mode where a crash mid-write leaves a half-complete file on the shared drive.

In [36]:
import pandas as pd

report_path = "production_report.xlsx"

with pd.ExcelWriter(report_path, engine="openpyxl") as writer:
    df_enriched.to_excel(writer, sheet_name="Transactions", index=False)
    rep_summary.reset_index().to_excel(writer, sheet_name="Rep Summary", index=False)
    df_quarantine.to_excel(writer, sheet_name="Quarantine Log", index=False)
    pivot.to_excel(writer, sheet_name="Pivot Analysis")

print(f"Wrote {report_path} with 4 sheets.")

Wrote production_report.xlsx with 4 sheets.


### 🧩 Practice 5.1
Add a fifth sheet called `"DQ Report"` containing the `dq_report` DataFrame from Module 4.
Use `mode="a"` with `if_sheet_exists="replace"` to add it to the existing file.

**Hints:**
- `pd.ExcelWriter(path, engine="openpyxl", mode="a", if_sheet_exists="replace")`

In [37]:
# TODO: add DQ Report sheet to production_report.xlsx

## 5.2 Programmatic styling with openpyxl

**Concept:** Once a workbook exists on disk, `openpyxl.load_workbook()` opens it for cell-level
formatting: `Font` (bold, color, size), `PatternFill` (background), `Alignment` (center, wrap),
`Border` objects, and `column_dimensions[col].width` for column widths. These are the exact same
controls as Excel's Format Cells dialog — just driven by code so they're reproducible.

**FAANG context:** Styled headers, currency formatting, and readable column widths are the
difference between a report that gets used and one that gets re-formatted by hand every cycle
(which means errors, every cycle).

In [38]:
from openpyxl import load_workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side, numbers

wb = load_workbook(report_path)

def style_header_row(ws, header_fill_color="1F4E78", font_color="FFFFFF"):
    # Apply bold white text on a dark background to the header row.
    header_font = Font(bold=True, color=font_color, size=11)
    header_fill = PatternFill(start_color=header_fill_color, end_color=header_fill_color, fill_type="solid")
    header_align = Alignment(horizontal="center", vertical="center", wrap_text=True)
    for cell in ws[1]:
        cell.font = header_font
        cell.fill = header_fill
        cell.alignment = header_align

def auto_column_widths(ws, padding=4):
    # Set column widths based on the longest cell value in each column.
    for col_cells in ws.iter_cols(min_row=1, max_row=min(ws.max_row, 200)):
        col_letter = col_cells[0].column_letter
        longest = max((len(str(c.value or "")) for c in col_cells), default=8)
        ws.column_dimensions[col_letter].width = min(longest + padding, 40)

def apply_zebra_striping(ws, even_color="F2F2F2", start_row=2):
    # Alternate row shading for readability (every other row gets a light fill).
    light_fill = PatternFill(start_color=even_color, end_color=even_color, fill_type="solid")
    for row_idx in range(start_row, ws.max_row + 1):
        if row_idx % 2 == 0:
            for cell in ws[row_idx]:
                cell.fill = light_fill

def apply_thin_borders(ws):
    thin = Border(*[Side(style="thin", color="D9D9D9")] * 4)
    for row in ws.iter_rows(min_row=1, max_row=ws.max_row):
        for cell in row:
            cell.border = thin

# Apply to the Rep Summary sheet
ws = wb["Rep Summary"]
style_header_row(ws)
auto_column_widths(ws)
apply_zebra_striping(ws)
apply_thin_borders(ws)

# Currency formatting on revenue columns (B=total_revenue, D=avg_order_value, E=max)
for row in ws.iter_rows(min_row=2, max_row=ws.max_row):
    for col_idx in [1, 3, 4]:  # 0-based: total_revenue, avg, max
        if col_idx < len(row):
            row[col_idx].number_format = "#,##0.00"

wb.save(report_path)
print("Styled Rep Summary: header, widths, zebra striping, borders, currency formats.")

Styled Rep Summary: header, widths, zebra striping, borders, currency formats.


### 🧩 Practice 5.2
Apply the same styling functions (`style_header_row`, `auto_column_widths`, `apply_thin_borders`)
to the `"Quarantine Log"` sheet. Additionally, make the `_quarantine_reason` column's cells use
a **red font** to draw attention.

**Hints:**
- Load the workbook, get `ws = wb["Quarantine Log"]`, apply the functions, then find the
  column index for `_quarantine_reason` and set `cell.font = Font(color="FF0000")`.

In [39]:
# TODO: style the Quarantine Log sheet

## 5.3 Live Excel formulas & conditional formatting

**Concept:** Writing a string starting with `=` into a cell makes it a live Excel formula.
Conditional formatting rules (from `openpyxl.formatting.rule`) apply visual styles automatically
based on cell values — exactly like Excel's Conditional Formatting menu, but driven by code.

**FAANG context:** Live formulas let stakeholders trust the numbers ("I can see the SUM
formula") and update them if they add a row. Conditional formatting (heatmaps, thresholds)
turns a wall of numbers into an instantly scannable report.

In [40]:
from openpyxl.formatting.rule import ColorScaleRule, CellIsRule
from openpyxl.styles import PatternFill as PF

ws = wb["Rep Summary"]
last_row = ws.max_row

# Add a formula-driven TOTAL row
total_row = last_row + 1
ws.cell(row=total_row, column=1, value="GRAND TOTAL").font = Font(bold=True, size=11)
ws.cell(row=total_row, column=2, value=f"=SUM(B2:B{last_row})").font = Font(bold=True)
ws.cell(row=total_row, column=2).number_format = "#,##0.00"
ws.cell(row=total_row, column=3, value=f"=SUM(C2:C{last_row})").font = Font(bold=True)
ws.cell(row=total_row, column=5, value=f"=MAX(E2:E{last_row})").font = Font(bold=True)
ws.cell(row=total_row, column=5).number_format = "#,##0.00"

# Color scale on the total_revenue column (red -> yellow -> green)
color_scale = ColorScaleRule(
    start_type="min", start_color="F8696B",
    mid_type="percentile", mid_value=50, mid_color="FFEB84",
    end_type="max", end_color="63BE7B",
)
ws.conditional_formatting.add(f"B2:B{last_row}", color_scale)

# Highlight cells in the avg_order_value column where value > 1000
pink_fill = PF(start_color="FFC7CE", end_color="FFC7CE", fill_type="solid")
highlight_rule = CellIsRule(operator="greaterThan", formula=["1000"], fill=pink_fill)
ws.conditional_formatting.add(f"D2:D{last_row}", highlight_rule)

wb.save(report_path)
print("Added TOTAL row, color scale, and conditional formatting to Rep Summary.")

Added TOTAL row, color scale, and conditional formatting to Rep Summary.


### 🧩 Practice 5.3
On the `"Transactions"` sheet, find the column for `line_revenue` and add a conditional
formatting rule that fills any cell with a value **less than 50** in a light yellow/amber
background, to flag unusually small transactions.

**Hints:**
- Iterate over `ws[1]` to find which column has the header `"line_revenue"`.
- `CellIsRule(operator="lessThan", formula=["50"], fill=your_fill)`

In [41]:
# TODO: add conditional formatting for small transactions

## 5.4 Automated Excel charts

**Concept:** `openpyxl.chart` builds native, editable Excel charts linked to worksheet data.
`BarChart`, `LineChart`, `PieChart` — you define `Reference` objects pointing at cell ranges,
attach them to the chart, and anchor it on a worksheet. The chart updates if the underlying
data changes.

**FAANG context:** Native Excel charts (not pasted images) are what stakeholders want — they can
click into them, adjust formatting, and copy them into presentations.

In [42]:
from openpyxl.chart import BarChart, LineChart, Reference

ws = wb["Rep Summary"]

# Bar chart: total revenue by rep
bar = BarChart()
bar.title = "Total Revenue by Sales Rep"
bar.x_axis.title = "Sales Rep"
bar.y_axis.title = "Revenue (USD)"
bar.style = 10
bar.width, bar.height = 18, 10

data_ref = Reference(ws, min_col=2, min_row=1, max_row=last_row)   # total_revenue col with header
cats_ref = Reference(ws, min_col=1, min_row=2, max_row=last_row)   # sales_rep names
bar.add_data(data_ref, titles_from_data=True)
bar.set_categories(cats_ref)
ws.add_chart(bar, "G2")

wb.save(report_path)
print("Added bar chart to Rep Summary sheet.")

Added bar chart to Rep Summary sheet.


## 5.5 Bonus: full report with xlsxwriter (single-pass)

**Concept:** `xlsxwriter` is write-only but often more concise for building a complete,
formatted report in a single pass. It's especially good when you're generating a report
from scratch every run rather than editing an existing file.

In [43]:
import xlsxwriter

with xlsxwriter.Workbook("xlsxwriter_report.xlsx") as xwb:
    # -- formats --
    hdr_fmt = xwb.add_format({"bold": True, "font_color": "white", "bg_color": "#1F4E78",
                               "align": "center", "border": 1})
    money_fmt = xwb.add_format({"num_format": "$#,##0.00", "border": 1})
    pct_fmt = xwb.add_format({"num_format": "0.0%", "border": 1})
    cell_fmt = xwb.add_format({"border": 1})
    zebra_fmt = xwb.add_format({"bg_color": "#F2F2F2", "border": 1})
    zebra_money = xwb.add_format({"bg_color": "#F2F2F2", "num_format": "$#,##0.00", "border": 1})

    # -- write Rep Summary --
    xws = xwb.add_worksheet("Summary")
    summary_data = rep_summary.reset_index()
    for c, col_name in enumerate(summary_data.columns):
        xws.write(0, c, col_name, hdr_fmt)
    for r, row_data in enumerate(summary_data.itertuples(index=False), start=1):
        fmt_base = zebra_fmt if r % 2 == 0 else cell_fmt
        fmt_money = zebra_money if r % 2 == 0 else money_fmt
        xws.write(r, 0, row_data.sales_rep, fmt_base)
        xws.write(r, 1, row_data.total_revenue, fmt_money)
        xws.write(r, 2, row_data.order_count, fmt_base)
        xws.write(r, 3, row_data.avg_order_value, fmt_money)
        xws.write(r, 4, row_data.unique_customers, fmt_base)
        xws.write(r, 5, row_data.max_single_order, fmt_money)

    xws.set_column(0, 0, 16)
    xws.set_column(1, 5, 14)

    # -- chart --
    chart = xwb.add_chart({"type": "column"})
    n = len(summary_data)
    chart.add_series({"name": "Total Revenue", "categories": ["Summary", 1, 0, n, 0],
                      "values": ["Summary", 1, 1, n, 1]})
    chart.set_title({"name": "Revenue by Rep (xlsxwriter)"})
    chart.set_size({"width": 600, "height": 400})
    xws.insert_chart("H2", chart)

print("Wrote xlsxwriter_report.xlsx (formatted, with chart, in one pass).")

Wrote xlsxwriter_report.xlsx (formatted, with chart, in one pass).


### 🧩 Practice 5.4
Using either openpyxl or xlsxwriter, create a **line chart** showing cumulative revenue over
time. You can use the `df_sorted` DataFrame from Module 3.3 (which has `order_date` and
`cumulative_revenue` columns). Write it to a new sheet in `production_report.xlsx` or create a
standalone file.

**Hints:**
- Write `df_sorted[["order_date", "cumulative_revenue"]]` to a sheet first.
- Then add a `LineChart` with `Reference` objects pointing at the data range.

In [44]:
# TODO: create a line chart of cumulative revenue

---
# Solutions Appendix

Complete solutions for every Practice Challenge. Try each one yourself first.

## Module 0 Solutions

In [45]:
# --- 0.1 ---
df_cust_sol = pd.read_csv("customer_master.csv")
profile_dataframe(df_cust_sol, "Customer Master")
# CustID: highest cardinality (100% unique), Country: lowest (~10 values)

  PROFILE: Customer Master
  Rows: 200  |  Columns: 5
  Total memory: 0.01 MB

     column   dtype  non_null  null_count  null_pct  n_unique  cardinality_pct           top_value                                              sample_values  mem_MB
     CustID     str       200           0       0.0       200            100.0           CUST-0001                    ['CUST-0001', 'CUST-0002', 'CUST-0003']  0.0032
CompanyName     str       200           0       0.0        20             10.0 Abstergo Industries ['Acme Corp', 'Globex International', 'Initech Solutions']  0.0044
    Segment     str       200           0       0.0         4              2.0                 SMB                    ['Government', 'Startup', 'Enterprise']  0.0029
    Country     str       200           0       0.0        10              5.0                  JP                                         ['KE', 'US', 'RW']  0.0019
CreditLimit float64       200           0       0.0       200            100.0             

,column,dtype,non_null,null_count,null_pct,n_unique,cardinality_pct,top_value,sample_values,mem_MB
0,CustID,str,200,0,0.0,200,100.0,CUST-0001,"['CUST-0001', 'CUST-0002', 'CUST-0003']",0.0032
1,CompanyName,str,200,0,0.0,20,10.0,Abstergo Industries,"['Acme Corp', 'Globex International', 'Initech...",0.0044
2,Segment,str,200,0,0.0,4,2.0,SMB,"['Government', 'Startup', 'Enterprise']",0.0029
3,Country,str,200,0,0.0,10,5.0,JP,"['KE', 'US', 'RW']",0.0019
4,CreditLimit,float64,200,0,0.0,200,100.0,5989.43,"[188440.62, 305801.54, 112588.38]",0.0015


In [46]:
# --- 0.2 ---
orig_nulls = df_raw[" Amount (USD)"].isna().sum()
new_nulls = df_clean_nulls[" Amount (USD)"].isna().sum()
amount_null_increase = new_nulls - orig_nulls
print(f"Amount null increase: {amount_null_increase}")

Amount null increase: 5


In [47]:
# --- 0.3 ---
import os
df_big_sol = pd.read_parquet("big_events.parquet")
mem_bytes = df_big_sol.memory_usage(deep=True).sum()
disk_bytes = os.path.getsize("big_events.parquet")
memory_to_disk_ratio = mem_bytes / disk_bytes
print(f"Memory: {mem_bytes/1024**2:.1f} MB, Disk: {disk_bytes/1024**2:.1f} MB, Ratio: {memory_to_disk_ratio:.2f}x")

Memory: 21.8 MB, Disk: 7.6 MB, Ratio: 2.86x


## Module 1 Solutions

In [48]:
# --- 1.1 ---
def load_all_sheets(path):
    wb_meta = load_workbook(path, read_only=True)
    sheet_names = wb_meta.sheetnames
    wb_meta.close()
    dfs = []
    for sheet in sheet_names:
        hdr = detect_header_row(path, sheet)
        df = pd.read_excel(path, sheet_name=sheet, header=hdr)
        df["_source_sheet"] = sheet
        dfs.append(df)
    return pd.concat(dfs, ignore_index=True)

df_all_sol = load_all_sheets("raw_transactions.xlsx")
print(f"Shape: {df_all_sol.shape}, source sheets: {df_all_sol['_source_sheet'].unique().tolist()}")

Shape: (2000, 13), source sheets: ['Jan_Mar', 'Apr_Jun']


In [49]:
# --- 1.2 ---
df_id_country = safe_read_csv("customer_master.csv", usecols=["CustID", "Country"])
print(f"Columns: {df_id_country.columns.tolist()}, shape: {df_id_country.shape}")
assert len(df_id_country.columns) == 2

  Loaded customer_master.csv with encoding=utf-8, shape=(200, 2)
Columns: ['CustID', 'Country'], shape: (200, 2)


## Module 2 Solutions

In [50]:
# --- 2.2 ---
region_before = df_txn["region"].nunique()
df_txn["region"] = df_txn["region"].str.strip().str.title()
region_after = df_txn["region"].nunique()
print(f"Region unique values: {region_before} -> {region_after}")

Region unique values: 25 -> 5


In [51]:
# --- 2.3 ---
df_txn["is_month_end"] = df_txn["order_date"] == (df_txn["order_date"] + pd.tseries.offsets.BMonthEnd(0))
df_txn[["order_date", "is_month_end"]].dropna().head(10)

,order_date,is_month_end
0,2026-03-18,False
4,2026-03-23,False
8,2026-01-23,False
9,2026-01-21,False
16,2026-01-21,False
18,2026-02-07,False
21,2026-03-01,False
24,2026-01-27,False
29,2026-03-27,False
31,2026-01-25,False


In [52]:
# --- 2.5 ---
df_valid_z, df_quarantine_z = quarantine_outliers(df_txn, "amount_usd", method="zscore")
print(f"Z-score quarantine: {len(df_quarantine_z)} rows")
print(f"IQR quarantine:     {len(df_quarantine)} rows")

Quarantined 24 rows (1.2%)
Z-score quarantine: 24 rows
IQR quarantine:     39 rows


## Module 3 Solutions

In [53]:
# --- 3.1 ---
df_inner = df_valid.merge(df_prod, on="product_code", how="inner", validate="m:1")
print(f"Left join rows: {len(df_enriched)}, Inner join rows: {len(df_inner)}")
print(f"Unmatched product rows: {len(df_enriched) - len(df_inner)}")

Left join rows: 1961, Inner join rows: 1961
Unmatched product rows: 0


In [54]:
# --- 3.2 ---
category_region_summary = df_enriched.groupby(["category", "region"]).agg(
    total_qty=("qty", "sum"),
    total_revenue=("line_revenue", "sum"),
    avg_price=("unitprice", "mean"),
).round(2).sort_values("total_revenue", ascending=False)
category_region_summary

total_qty  total_revenue  avg_price
category        region                                      
Office Supplies East          2948     3046424.69    1028.08
                Central       2520     2521343.43    1040.35
                North         2232     2413718.23    1052.53
Software        North         1475     2052293.62    1387.40
Office Supplies West          2158     2007094.73     953.03
...                            ...            ...        ...
                 North           6        3217.27    1021.92
Services        South            6        2941.74     490.29
                East             9        2861.82     317.98
Office Supplies south            7        2053.52     293.36
Services        north            1         208.44     208.44

[102 rows x 3 columns]

In [55]:
# --- 3.3 ---
df_enriched["rep_order_rank"] = df_enriched.groupby("sales_rep")["line_revenue"].rank(
    method="dense", ascending=False
)
df_enriched[["sales_rep", "line_revenue", "rep_order_rank"]].sort_values(
    ["sales_rep", "rep_order_rank"]).head(10)

,sales_rep,line_revenue,rep_order_rank
1545,Alice,63713.25,1.0
1849,Alice,47164.95,2.0
1863,Alice,35028.00,3.0
667,Alice,29318.22,4.0
897,Alice,21908.50,5.0
1920,Alice,17111.69,6.0
666,Alice,16195.41,7.0
68,Alice,7005.60,8.0
1676,Alice,6490.43,9.0
1280,Alice,5519.90,10.0


In [56]:
# --- 3.4 ---
pivot_month_status = pd.pivot_table(
    df_enriched, values="txn_id", index="order_month", columns="status",
    aggfunc="count", fill_value=0,
)
pivot_month_status.columns = [str(c) for c in pivot_month_status.columns]
pivot_month_status

,Cancelled,Completed,Pending,Refunded
order_month,,,,
2026-01,23,26,28,19
2026-02,19,20,19,21
2026-03,17,24,18,23
2026-04,16,20,25,18
2026-05,21,19,19,20
2026-06,14,25,14,16


## Module 4 Solutions

In [57]:
# --- 4.1 ---
quality_checks_expanded = quality_checks + [
    {"name": "status values are valid",
     "check": lambda df: set(df["status"].dropna().unique()) <= {"Completed", "Pending", "Refunded", "Cancelled"},
     "severity": "warn"},
    {"name": "customer_id format check",
     "check": lambda df: df["customer_id"].dropna().str.match(r"^CUST-\d{4}$").all(),
     "severity": "error"},
]
print("Expanded DQ checks:")
run_data_quality_checks(df_valid, quality_checks_expanded)

Expanded DQ checks:
  [X] txn_id is unique -> FAIL
  [X] order_date has no nulls -> WARN
  [+] qty is positive -> PASS
  [+] amount_usd is non-negative -> PASS
  [X] region in valid set -> WARN
  [+] row count > 100 -> PASS
  [+] status values are valid -> PASS
  [+] customer_id format check -> PASS


(False,
                         check status severity
 0            txn_id is unique   FAIL    error
 1     order_date has no nulls   WARN     warn
 2             qty is positive   PASS    error
 3  amount_usd is non-negative   PASS    error
 4         region in valid set   WARN     warn
 5             row count > 100   PASS    error
 6     status values are valid   PASS     warn
 7    customer_id format check   PASS    error)

In [58]:
# --- 4.2 ---
conditions = [
    df_enriched["amount_usd"] < 50,
    df_enriched["amount_usd"] < 500,
    df_enriched["amount_usd"] < 5000,
]
choices = ["Micro", "Small", "Medium"]
df_enriched["amount_bucket"] = np.select(conditions, choices, default="Large")
print(df_enriched["amount_bucket"].value_counts())

amount_bucket
Medium    1329
Large      469
Small      143
Micro       20
Name: count, dtype: int64


In [59]:
# --- 4.3 ---
df_window_result = duckdb.sql('''
    SELECT event_id, user_id, event_type, amount, ts,
           SUM(amount) OVER (PARTITION BY user_id ORDER BY ts) AS running_total
    FROM df_big
    LIMIT 10
''').df()
df_window_result

,event_id,user_id,event_type,amount,ts,running_total
0,9542,53,click,10.32,2026-01-07 15:02:00,10.32
1,10135,53,purchase,19.04,2026-01-08 00:55:00,29.36
2,115848,53,logout,12.61,2026-03-22 10:48:00,41.97
3,118676,53,logout,34.60,2026-03-24 09:56:00,76.57
4,136332,53,view,125.86,2026-04-05 16:12:00,202.43
5,157518,53,signup,35.57,2026-04-20 09:18:00,238.00
6,162342,53,click,31.75,2026-04-23 17:42:00,269.75
7,173463,53,signup,8.12,2026-05-01 11:03:00,277.87
8,175178,53,purchase,76.15,2026-05-02 15:38:00,354.02
9,178016,53,click,54.03,2026-05-04 14:56:00,408.05


## Module 5 Solutions

In [60]:
# --- 5.1 ---
with pd.ExcelWriter(report_path, engine="openpyxl", mode="a", if_sheet_exists="replace") as writer:
    dq_report.to_excel(writer, sheet_name="DQ Report", index=False)
print("Added DQ Report sheet.")

Added DQ Report sheet.


In [61]:
# --- 5.2 ---
wb_sol = load_workbook(report_path)
ws_q = wb_sol["Quarantine Log"]
style_header_row(ws_q)
auto_column_widths(ws_q)
apply_thin_borders(ws_q)

# Find _quarantine_reason column and make it red
for cell in ws_q[1]:
    if cell.value == "_quarantine_reason":
        reason_col = cell.column
        for row_idx in range(2, ws_q.max_row + 1):
            ws_q.cell(row=row_idx, column=reason_col).font = Font(color="FF0000")
        break

wb_sol.save(report_path)
print("Styled Quarantine Log sheet.")

Styled Quarantine Log sheet.


In [62]:
# --- 5.3 ---
wb_sol = load_workbook(report_path)
ws_txn = wb_sol["Transactions"]
# Find line_revenue column
rev_col = None
for cell in ws_txn[1]:
    if cell.value == "line_revenue":
        rev_col = cell.column_letter
        break

if rev_col:
    amber_fill = PF(start_color="FFEB84", end_color="FFEB84", fill_type="solid")
    small_rule = CellIsRule(operator="lessThan", formula=["50"], fill=amber_fill)
    last = ws_txn.max_row
    ws_txn.conditional_formatting.add(f"{rev_col}2:{rev_col}{last}", small_rule)
    wb_sol.save(report_path)
    print(f"Added conditional formatting to {rev_col}2:{rev_col}{last}")

Added conditional formatting to Z2:Z1962


In [63]:
# --- 5.4 ---
from openpyxl.chart import LineChart as LC, Reference as LCRef

# Write cumulative revenue data to a new sheet
cum_data = df_sorted[["order_date", "cumulative_revenue"]].head(200).copy()
cum_data["order_date"] = cum_data["order_date"].dt.strftime("%Y-%m-%d")

wb_sol = load_workbook(report_path)
ws_cum = wb_sol.create_sheet("Cumulative Revenue")
ws_cum.append(["Date", "Cumulative Revenue"])
for _, row in cum_data.iterrows():
    ws_cum.append([row["order_date"], row["cumulative_revenue"]])

line_chart = LC()
line_chart.title = "Cumulative Revenue Over Time"
line_chart.x_axis.title = "Date"
line_chart.y_axis.title = "Cumulative Revenue (USD)"
data = LCRef(ws_cum, min_col=2, min_row=1, max_row=ws_cum.max_row)
cats = LCRef(ws_cum, min_col=1, min_row=2, max_row=ws_cum.max_row)
line_chart.add_data(data, titles_from_data=True)
line_chart.set_categories(cats)
line_chart.width, line_chart.height = 20, 10
ws_cum.add_chart(line_chart, "D2")

wb_sol.save(report_path)
print("Added Cumulative Revenue sheet with line chart.")

Added Cumulative Revenue sheet with line chart.


---
## Where to go from here

You now have a production toolkit covering: **profiling** (hidden nulls, cardinality, memory),
**ingestion** (encoding fallback, header detection, chunking), **cleaning** (fuzzy matching,
datetime coercion, type optimization, quarantine), **transforms** (merge/join, named aggregation,
window functions, DuckDB), **quality** (assertion frameworks, vectorized logic), and **export**
(multi-sheet, formatted, charted workbooks).

Next steps worth exploring: `great_expectations` for declarative data contracts, `Polars` as a
faster-than-pandas DataFrame library, `dbt` for SQL-based transform orchestration, and `Airflow`
or `Prefect` for scheduling these pipelines to run automatically.